<a href="https://colab.research.google.com/github/eshwar-7419/cads/blob/main/CARC_IDS_Phase2B_v3_Validated_Continual_IDS_Benchmark_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2B-v3 — Validated Continual IDS Benchmark

## Why this version exists

The Phase-2B-v2 results exposed a second protocol problem:

- E4 and E5 contain only attack traffic.
- Therefore their original evaluation sets cannot measure IDS false-positive rate.
- Blind continual training on attack-only windows can collapse the benign decision boundary.

That failure is useful as a stress case, but it must not be mistaken for a complete continual-learning benchmark.

This version separates **adaptation data** from **operational evaluation**.

---

# Final stream

The evidence-based sequential stream remains:

| Experience | Segments | Role |
|---|---|---|
| E1 | 3 | Initial attack-aware detector |
| E2 | 4–6 | High-attack regime |
| E3 | 7 | Attack-family transition |
| E4 | 8 | Generic-dominant regime |
| E5 | 9–10 | Stable late regime |

Segments 1–2 contain benign traffic and are reserved as a **fixed benign evaluation reference only**.

They are never used for model adaptation.

---

# Evaluation design

For each experience:

```text
Current attack evaluation samples
                +
Fixed benign reference samples
                ↓
       Operational evaluation
```

This makes:

- TP measurable;
- TN measurable;
- FP measurable;
- FN measurable;
- FPR meaningful at every experience.

The benign reference is selected once and reused unchanged.

This is deliberately a stress-test evaluation set, not a claim that every real-world deployment window has the same class balance.

---

# Adaptation baselines

### 1. Static LightGBM

Train once on E1.

Never adapt.

### 2. Blind continual LightGBM

Train/update only on the current experience.

This is intentionally unsafe under attack-heavy windows.

### 3. Replay LightGBM

Current experience + bounded historical replay.

Replay is class-balanced so that benign discrimination is retained.

### 4. EWC MLP

Parameter-regularized neural continual learner.

This is a reference baseline, not EWC-LightGBM.

---

# Important distinction

The blind baseline is allowed to fail.

That failure answers:

> What happens if an IDS blindly learns from an attack-heavy current window?

The replay baseline answers:

> Can bounded historical memory prevent this failure?

The future proposed controller will answer:

> Can the system decide whether adaptation is necessary and choose a cheaper/safer adaptation action based on drift, detection degradation, and resources?

That final question is Phase 2C.

---

# Threshold protocol

Every method gets one threshold selected from its own E1 calibration data.

Threshold selection:

- threshold range: 0.05–0.95;
- FPR constraint: <= 10%;
- maximize F1 under that constraint;
- if no threshold satisfies the constraint, minimize FPR then maximize F1;
- freeze threshold for all later experiences and final test.

No later experience or final test is used for threshold tuning.

---

# Final independent test

The original temporal test set is never used to construct the continual stream, replay memory, benign evaluation reference, or thresholds.


In [1]:
# 1. Install dependencies
!pip -q install datasets lightgbm psutil joblib scikit-learn torch

print("Dependencies installed.")


Dependencies installed.


In [2]:
# 2. Imports and reproducibility

import os
import gc
import json
import time
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil
import joblib

from datasets import load_dataset

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, average_precision_score
)
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE = Path("/content/carc_ids_phase2b_v3")
RESULTS = BASE / "results"
ARTIFACTS = BASE / "artifacts"

RESULTS.mkdir(parents=True, exist_ok=True)
ARTIFACTS.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Working directory:", BASE)


Device: cpu
Working directory: /content/carc_ids_phase2b_v3


# 3. Load the same dataset

Dataset:

`lacg030175/UNSW-NB15`

Configuration:

`temporal`

Only the training split constructs the continual stream.


In [3]:
# 3. Load dataset

ds = load_dataset(
    "lacg030175/UNSW-NB15",
    "temporal"
)

train_df = ds["train"].to_pandas()
test_df = ds["test"].to_pandas()

print("Train:", train_df.shape)
print("Test :", test_df.shape)

print("\nTraining labels:")
display(train_df["label"].value_counts().sort_index().rename("count").to_frame())

print("\nTest labels:")
display(test_df["label"].value_counts().sort_index().rename("count").to_frame())


README.md:   0%|          | 0.00/5.32k [00:00<?, ?B/s]

temporal/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 12.7MB            

temporal/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

temporal/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.25MB            

temporal/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating test split:   0%|          | 0/82332 [00:00<?, ? examples/s]

Train: (175341, 44)
Test : (82332, 44)

Training labels:


,count
label,
0,56000
1,119341



Test labels:


,count
label,
0,37000
1,45332


# 4. Reconstruct the ten temporal segments


In [4]:
# 4. Segmentation

N_SEGMENTS = 10
segment_indices = np.array_split(np.arange(len(train_df)), N_SEGMENTS)

segments = {
    sid: train_df.iloc[idx].copy()
    for sid, idx in enumerate(segment_indices, start=1)
}

EXPERIENCES = {
    "E1_initial_attack": [3],
    "E2_high_attack": [4, 5, 6],
    "E3_attack_transition": [7],
    "E4_generic_dominant": [8],
    "E5_stable_late": [9, 10]
}

experience_dfs = {
    name: pd.concat(
        [segments[s] for s in sids],
        ignore_index=True
    )
    for name, sids in EXPERIENCES.items()
}

summary_rows = []

for name, df in experience_dfs.items():
    summary_rows.append({
        "experience": name,
        "segments": ",".join(map(str, EXPERIENCES[name])),
        "rows": len(df),
        "benign": int((df.label == 0).sum()),
        "attack": int((df.label == 1).sum()),
        "attack_pct": float(df.label.mean()),
        "attack_categories": int(
            df.loc[df.label == 1, "attack_cat"].nunique()
        )
    })

experience_summary = pd.DataFrame(summary_rows)

display(experience_summary)

experience_summary.to_csv(
    RESULTS / "experience_summary.csv",
    index=False
)


,experience,segments,rows,benign,attack,attack_pct,attack_categories
0,E1_initial_attack,3,17534,12842,4692,0.267594,8
1,E2_high_attack,"4,5,6",52602,5405,47197,0.897247,8
2,E3_attack_transition,7,17534,2684,14850,0.846926,9
3,E4_generic_dominant,8,17534,0,17534,1.000000,9
4,E5_stable_late,"9,10",35068,0,35068,1.000000,9


# 5. Reserve Segments 1–2 as a benign evaluation reference

Segments 1–2 contain no attack examples in the previously established temporal stream.

We use a fixed sample of benign rows from these segments for **evaluation only**.

They are never:

- used for E1 training;
- used for adaptation;
- placed into replay memory;
- used for threshold calibration.

A fixed reference is necessary so FPR is comparable across E1–E5.


In [5]:
# 5. Fixed benign evaluation reference

benign_reference_source = pd.concat(
    [segments[1], segments[2]],
    ignore_index=True
)

benign_reference_source = benign_reference_source[
    benign_reference_source["label"] == 0
].copy()

BENIGN_REFERENCE_SIZE = min(10000, len(benign_reference_source))

benign_reference_df = benign_reference_source.sample(
    n=BENIGN_REFERENCE_SIZE,
    random_state=SEED
).reset_index(drop=True)

print("Benign reference source:", len(benign_reference_source))
print("Fixed benign evaluation reference:", len(benign_reference_df))

assert benign_reference_df["label"].eq(0).all()

# This reference must not enter adaptation.


Benign reference source: 35069
Fixed benign evaluation reference: 10000


# 6. Leakage-safe preprocessing

The preprocessing transformer is fitted on the supplied training split only.

The temporal test is not used to fit preprocessing.


In [6]:
# 6. Feature preparation

DROP = [
    c for c in ["label", "attack_cat", "id", "ID", "index"]
    if c in train_df.columns
]

X_train_raw = train_df.drop(columns=DROP, errors="ignore").copy()
X_test_raw = test_df.drop(columns=DROP, errors="ignore").copy()

y_test = test_df["label"].astype(int).to_numpy()

numeric_cols = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [
    c for c in X_train_raw.columns
    if c not in numeric_cols
]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), numeric_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]), categorical_cols)
])

# Fit only on training split.
preprocessor.fit(X_train_raw)

joblib.dump(
    preprocessor,
    ARTIFACTS / "preprocessor.joblib"
)

print("Processed feature dimension:",
      preprocessor.transform(X_train_raw.iloc[:1]).shape[1])


Processed feature dimension: 194


In [7]:
# 7. Transformation helper

def transform_df(df):
    X = df.drop(columns=DROP, errors="ignore").copy()
    y = df["label"].astype(int).to_numpy()

    Xp = preprocessor.transform(X).astype(np.float32)

    return Xp, y


# 8. Correct E1 initialization

E1 is stratified:

- 56% training;
- 14% calibration;
- 30% evaluation.

This ensures the initial detector sees both benign and attack traffic.

The E1 evaluation portion is combined with the fixed benign reference only for operational evaluation.


In [8]:
# 8. E1 stratified partition

e1_df = experience_dfs["E1_initial_attack"].copy()

e1_train_df, e1_holdout_df = train_test_split(
    e1_df,
    test_size=0.44,
    stratify=e1_df["label"],
    random_state=SEED
)

e1_cal_df, e1_eval_attack_df = train_test_split(
    e1_holdout_df,
    test_size=(0.30 / 0.44),
    stratify=e1_holdout_df["label"],
    random_state=SEED
)

print("E1 total:", len(e1_df))
print("E1 train:", len(e1_train_df))
print("E1 calibration:", len(e1_cal_df))
print("E1 attack evaluation:", len(e1_eval_attack_df))

for name, df in [
    ("E1 train", e1_train_df),
    ("E1 calibration", e1_cal_df),
    ("E1 attack evaluation", e1_eval_attack_df)
]:
    print(
        name,
        "benign =", int((df.label == 0).sum()),
        "attack =", int((df.label == 1).sum()),
        "attack_pct =", round(float(df.label.mean()), 4)
    )

assert e1_train_df.label.nunique() == 2
assert e1_cal_df.label.nunique() == 2
assert e1_eval_attack_df.label.nunique() == 2


E1 total: 17534
E1 train: 9819
E1 calibration: 2454
E1 attack evaluation: 5261
E1 train benign = 7191 attack = 2628 attack_pct = 0.2676
E1 calibration benign = 1797 attack = 657 attack_pct = 0.2677
E1 attack evaluation benign = 3854 attack = 1407 attack_pct = 0.2674


In [9]:
# 9. Transform E1

E1_X_train, E1_y_train = transform_df(e1_train_df)
E1_X_cal, E1_y_cal = transform_df(e1_cal_df)
E1_X_attack_eval, E1_y_attack_eval = transform_df(e1_eval_attack_df)

X_benign_ref, y_benign_ref = transform_df(
    benign_reference_df
)

assert np.all(y_benign_ref == 0)

print("E1 train:", E1_X_train.shape)
print("E1 calibration:", E1_X_cal.shape)
print("E1 attack evaluation:", E1_X_attack_eval.shape)
print("Benign reference:", X_benign_ref.shape)


E1 train: (9819, 194)
E1 calibration: (2454, 194)
E1 attack evaluation: (5261, 194)
Benign reference: (10000, 194)


# 10. Build E2–E5 adaptation and attack-evaluation partitions

E2–E5 are attack-heavy or attack-only.

We therefore split **attack observations only**:

- first 70% → adaptation;
- final 30% → attack evaluation.

The fixed benign reference is added only to evaluation.

This prevents us from silently treating attack-only windows as complete operational traffic distributions.


In [10]:
# 10. Stream data

stream_data = {
    "E1_initial_attack": {
        "X_adapt": E1_X_train,
        "y_adapt": E1_y_train,
        "X_cal": E1_X_cal,
        "y_cal": E1_y_cal,
        "X_attack_eval": E1_X_attack_eval,
        "y_attack_eval": E1_y_attack_eval
    }
}

for name in list(EXPERIENCES.keys())[1:]:
    df = experience_dfs[name].copy()

    attack_df = df[df["label"] == 1].copy()

    assert len(attack_df) > 0

    split = int(len(attack_df) * 0.70)
    split = max(1, min(split, len(attack_df) - 1))

    adapt_df = attack_df.iloc[:split].copy()
    attack_eval_df = attack_df.iloc[split:].copy()

    X_adapt, y_adapt = transform_df(adapt_df)
    X_attack_eval, y_attack_eval = transform_df(attack_eval_df)

    stream_data[name] = {
        "X_adapt": X_adapt,
        "y_adapt": y_adapt,
        "X_attack_eval": X_attack_eval,
        "y_attack_eval": y_attack_eval
    }

    print(
        name,
        "adapt attacks:", len(y_adapt),
        "eval attacks:", len(y_attack_eval)
    )


E2_high_attack adapt attacks: 33037 eval attacks: 14160
E3_attack_transition adapt attacks: 10395 eval attacks: 4455
E4_generic_dominant adapt attacks: 12273 eval attacks: 5261
E5_stable_late adapt attacks: 24547 eval attacks: 10521


# 11. Operational evaluation builder

For every experience:

```text
current attack evaluation
+
fixed benign reference
```

The benign reference is the same for every experience.

This makes FPR comparable across the stream.

Because the reference is fixed, this should be interpreted as a **controlled operational stress test**, not as the empirical class ratio of the original segment.


In [11]:
# 11. Build operational evaluation sets

def make_operational_eval(data):
    X_attack = data["X_attack_eval"]
    y_attack = data["y_attack_eval"]

    # Keep every current attack evaluation sample.
    # Match the benign reference size to attack count when possible.
    n_benign = min(len(X_benign_ref), len(X_attack))

    X_b = X_benign_ref[:n_benign]
    y_b = y_benign_ref[:n_benign]

    X_eval = np.concatenate([X_b, X_attack], axis=0)
    y_eval = np.concatenate([y_b, y_attack], axis=0)

    # Fixed deterministic ordering.
    return X_eval, y_eval

for name, data in stream_data.items():
    X_eval, y_eval = make_operational_eval(data)

    data["X_operational_eval"] = X_eval
    data["y_operational_eval"] = y_eval

    print(
        name,
        "operational eval:",
        X_eval.shape,
        "attack rate:",
        round(float(y_eval.mean()), 4)
    )


E1_initial_attack operational eval: (10522, 194) attack rate: 0.1337
E2_high_attack operational eval: (24160, 194) attack rate: 0.5861
E3_attack_transition operational eval: (8910, 194) attack rate: 0.5
E4_generic_dominant operational eval: (10522, 194) attack rate: 0.5
E5_stable_late operational eval: (20521, 194) attack rate: 0.5127


# 12. Metrics and threshold selection


In [12]:
# 12. Metrics

def binary_metrics(y_true, probs, threshold):
    pred = (probs >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, pred, labels=[0, 1]
    ).ravel()

    return {
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "fpr": fp / (fp + tn) if (fp + tn) else np.nan,
        "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "balanced_accuracy": (
            (
                tp / (tp + fn) if (tp + fn) else 0
            ) +
            (
                tn / (tn + fp) if (tn + fp) else 0
            )
        ) / 2,
        "roc_auc": roc_auc_score(y_true, probs),
        "pr_auc": average_precision_score(y_true, probs),
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn)
    }

def select_threshold(y_true, probs, fpr_limit=0.10):
    rows = []

    for threshold in np.linspace(0.05, 0.95, 181):
        m = binary_metrics(
            y_true,
            probs,
            threshold
        )

        rows.append({
            "threshold": threshold,
            "f1": m["f1"],
            "recall": m["recall"],
            "precision": m["precision"],
            "fpr": m["fpr"]
        })

    table = pd.DataFrame(rows)

    feasible = table[
        table["fpr"] <= fpr_limit
    ]

    if len(feasible):
        best = feasible.sort_values(
            ["f1", "recall", "fpr"],
            ascending=[False, False, True]
        ).iloc[0]
        reason = "best_f1_under_fpr_constraint"
    else:
        best = table.sort_values(
            ["fpr", "f1"],
            ascending=[True, False]
        ).iloc[0]
        reason = "minimum_fpr_fallback"

    return float(best["threshold"]), table, reason


# 13. Resource helpers


In [13]:
# 13. Resource measurement

def rss_mb():
    return psutil.Process(
        os.getpid()
    ).memory_info().rss / (1024**2)

def resource_snapshot():
    return {
        "rss_mb": rss_mb(),
        "cpu_percent": psutil.cpu_percent(interval=0.1)
    }


# 14. Static LightGBM

Train once on E1.

Threshold is calibrated on E1 calibration.

Then evaluate every experience without adaptation.


In [14]:
# 14. Static baseline

static_model = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1
)

t0 = time.perf_counter()
static_model.fit(E1_X_train, E1_y_train)
static_train_time = time.perf_counter() - t0

static_cal_probs = static_model.predict_proba(
    E1_X_cal
)[:, 1]

static_threshold, static_threshold_table, static_reason = select_threshold(
    E1_y_cal,
    static_cal_probs
)

static_threshold_table.to_csv(
    RESULTS / "static_threshold_selection.csv",
    index=False
)

print("Static threshold:", static_threshold)
print("Reason:", static_reason)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Static threshold: 0.4499999999999999
Reason: best_f1_under_fpr_constraint


In [15]:
# 15. Static stream evaluation

static_rows = []

for exp_name, data in stream_data.items():

    probs = static_model.predict_proba(
        data["X_operational_eval"]
    )[:, 1]

    m = binary_metrics(
        data["y_operational_eval"],
        probs,
        static_threshold
    )

    m.update({
        "method": "Static_LightGBM",
        "model_state": "E1",
        "evaluated_experience": exp_name,
        "threshold": static_threshold,
        "train_time_sec": static_train_time
    })

    static_rows.append(m)

static_stream = pd.DataFrame(static_rows)

display(
    static_stream[
        ["method","evaluated_experience","f1",
         "recall","precision","fpr",
         "specificity","balanced_accuracy"]
    ].round(4)
)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,method,evaluated_experience,f1,recall,precision,fpr,specificity,balanced_accuracy
0,Static_LightGBM,E1_initial_attack,0.8577,0.8436,0.8722,0.0191,0.9809,0.9123
1,Static_LightGBM,E2_high_attack,0.8987,0.8209,0.9927,0.0085,0.9915,0.9062
2,Static_LightGBM,E3_attack_transition,0.8790,0.7906,0.9896,0.0083,0.9917,0.8911
3,Static_LightGBM,E4_generic_dominant,0.8470,0.7405,0.9891,0.0082,0.9918,0.8662
4,Static_LightGBM,E5_stable_late,0.8541,0.7514,0.9894,0.0085,0.9915,0.8715


# 16. Blind continual LightGBM

This intentionally uses **only the current adaptation window**.

It represents the naive deployment mistake:

> "The latest window is the new training distribution."

Because E4/E5 are attack-only, this baseline is expected to expose the risk of catastrophic benign-discrimination loss.

We retain it because the failure itself is a meaningful stress-test result.


In [16]:
# 16. Blind continual baseline

naive_model = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1
)

naive_rows = []
naive_threshold = None

for i, (exp_name, data) in enumerate(
    stream_data.items(),
    start=1
):

    t0 = time.perf_counter()

    naive_model.fit(
        data["X_adapt"],
        data["y_adapt"]
    )

    train_time = time.perf_counter() - t0

    if i == 1:
        cal_probs = naive_model.predict_proba(
            data["X_cal"]
        )[:, 1]

        naive_threshold, naive_threshold_table, naive_reason = select_threshold(
            data["y_cal"],
            cal_probs
        )

        naive_threshold_table.to_csv(
            RESULTS / "naive_threshold_selection.csv",
            index=False
        )

    for eval_name, eval_data in list(
        stream_data.items()
    )[:i]:

        probs = naive_model.predict_proba(
            eval_data["X_operational_eval"]
        )[:, 1]

        m = binary_metrics(
            eval_data["y_operational_eval"],
            probs,
            naive_threshold
        )

        m.update({
            "method": "Blind_CL_LightGBM",
            "after_experience": exp_name,
            "evaluated_experience": eval_name,
            "threshold": naive_threshold,
            "train_time_sec": train_time,
            "rss_mb": rss_mb()
        })

        naive_rows.append(m)

naive_stream = pd.DataFrame(naive_rows)

display(
    naive_stream[
        ["method","after_experience",
         "evaluated_experience","f1",
         "recall","precision","fpr"]
    ].round(4)
)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

,method,after_experience,evaluated_experience,f1,recall,precision,fpr
0,Blind_CL_LightGBM,E1_initial_attack,E1_initial_attack,0.8577,0.8436,0.8722,0.0191
1,Blind_CL_LightGBM,E2_high_attack,E1_initial_attack,0.0000,0.0000,0.0000,0.0000
2,Blind_CL_LightGBM,E2_high_attack,E2_high_attack,0.0000,0.0000,0.0000,0.0000
3,Blind_CL_LightGBM,E3_attack_transition,E1_initial_attack,0.0000,0.0000,0.0000,0.0000
4,Blind_CL_LightGBM,E3_attack_transition,E2_high_attack,0.0000,0.0000,0.0000,0.0000
5,Blind_CL_LightGBM,E3_attack_transition,E3_attack_transition,0.0000,0.0000,0.0000,0.0000
6,Blind_CL_LightGBM,E4_generic_dominant,E1_initial_attack,0.0000,0.0000,0.0000,0.0000
7,Blind_CL_LightGBM,E4_generic_dominant,E2_high_attack,0.0000,0.0000,0.0000,0.0000
8,Blind_CL_LightGBM,E4_generic_dominant,E3_attack_transition,0.0000,0.0000,0.0000,0.0000
9,Blind_CL_LightGBM,E4_generic_dominant,E4_generic_dominant,0.0000,0.0000,0.0000,0.0000


# 17. Replay-protected LightGBM

Replay stores historical samples from previous adaptation windows.

The replay buffer is **class-balanced** whenever possible.

This is critical because otherwise an attack-heavy stream can make replay itself attack-dominated.

Replay budget:

`2000 samples per experience`

The replay set is bounded and therefore its memory cost is measurable.


In [17]:
# 17. Replay baseline

REPLAY_PER_EXPERIENCE = 2000
rng = np.random.default_rng(SEED)

replay_model = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1
)

replay_X = []
replay_y = []
replay_rows = []
replay_threshold = None

def balanced_sample(X, y, n, rng):
    y = np.asarray(y)

    classes = np.unique(y)

    if len(classes) < 2:
        idx = rng.choice(
            len(y),
            size=min(n, len(y)),
            replace=False
        )
        return X[idx], y[idx]

    per_class = max(1, n // len(classes))

    chosen = []

    for c in classes:
        idx_c = np.where(y == c)[0]
        k = min(per_class, len(idx_c))

        if k:
            chosen.append(
                rng.choice(
                    idx_c,
                    size=k,
                    replace=False
                )
            )

    chosen = np.concatenate(chosen)

    if len(chosen) > n:
        chosen = rng.choice(
            chosen,
            size=n,
            replace=False
        )

    return X[chosen], y[chosen]

for i, (exp_name, data) in enumerate(
    stream_data.items(),
    start=1
):

    if replay_X:
        X_old = np.concatenate(
            replay_X,
            axis=0
        )
        y_old = np.concatenate(
            replay_y,
            axis=0
        )

        X_fit = np.concatenate(
            [data["X_adapt"], X_old],
            axis=0
        )

        y_fit = np.concatenate(
            [data["y_adapt"], y_old],
            axis=0
        )

    else:
        X_fit = data["X_adapt"]
        y_fit = data["y_adapt"]

    t0 = time.perf_counter()

    replay_model.fit(
        X_fit,
        y_fit
    )

    train_time = time.perf_counter() - t0

    if i == 1:
        cal_probs = replay_model.predict_proba(
            data["X_cal"]
        )[:, 1]

        replay_threshold, replay_threshold_table, replay_reason = select_threshold(
            data["y_cal"],
            cal_probs
        )

        replay_threshold_table.to_csv(
            RESULTS / "replay_threshold_selection.csv",
            index=False
        )

    for eval_name, eval_data in list(
        stream_data.items()
    )[:i]:

        probs = replay_model.predict_proba(
            eval_data["X_operational_eval"]
        )[:, 1]

        m = binary_metrics(
            eval_data["y_operational_eval"],
            probs,
            replay_threshold
        )

        m.update({
            "method": "Replay_LightGBM",
            "after_experience": exp_name,
            "evaluated_experience": eval_name,
            "threshold": replay_threshold,
            "train_time_sec": train_time,
            "replay_size": sum(
                len(x) for x in replay_X
            ),
            "rss_mb": rss_mb()
        })

        replay_rows.append(m)

    X_new, y_new = balanced_sample(
        data["X_adapt"],
        data["y_adapt"],
        REPLAY_PER_EXPERIENCE,
        rng
    )

    replay_X.append(X_new)
    replay_y.append(y_new)

replay_stream = pd.DataFrame(replay_rows)

display(
    replay_stream[
        ["method","after_experience",
         "evaluated_experience",
         "f1","recall","precision",
         "fpr","replay_size"]
    ].round(4)
)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

,method,after_experience,evaluated_experience,f1,recall,precision,fpr,replay_size
0,Replay_LightGBM,E1_initial_attack,E1_initial_attack,0.8577,0.8436,0.8722,0.0191,0
1,Replay_LightGBM,E2_high_attack,E1_initial_attack,0.6124,1.0000,0.4413,0.1954,2000
2,Replay_LightGBM,E2_high_attack,E2_high_attack,0.9922,0.9996,0.9849,0.0217,2000
3,Replay_LightGBM,E3_attack_transition,E1_initial_attack,0.6434,0.9936,0.4757,0.1691,4000
4,Replay_LightGBM,E3_attack_transition,E2_high_attack,0.9895,0.9937,0.9854,0.0209,4000
5,Replay_LightGBM,E3_attack_transition,E3_attack_transition,0.9877,0.9969,0.9786,0.0218,4000
6,Replay_LightGBM,E4_generic_dominant,E1_initial_attack,0.6502,0.9936,0.4832,0.1640,6000
7,Replay_LightGBM,E4_generic_dominant,E2_high_attack,0.9880,0.9907,0.9853,0.0210,6000
8,Replay_LightGBM,E4_generic_dominant,E3_attack_transition,0.9881,0.9975,0.9789,0.0215,6000
9,Replay_LightGBM,E4_generic_dominant,E4_generic_dominant,0.9879,0.9983,0.9777,0.0228,6000


# 18. EWC MLP reference baseline

EWC is implemented with a neural network because EWC requires parameter-level regularization.

The current adaptation data is used as the learning signal.

This baseline is intentionally separate from replay.


In [18]:
# 18. EWC model

class SmallMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        hidden = min(
            128,
            max(32, input_dim // 4)
        )

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(hidden, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

def make_loader(
    X,
    y,
    batch_size=256,
    shuffle=True
):
    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32)
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle
    )

def train_ewc(
    model,
    X,
    y,
    old_params=None,
    fisher=None,
    ewc_lambda=100.0,
    epochs=5,
    lr=1e-3
):
    model.train()

    loader = make_loader(X, y)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    criterion = nn.BCEWithLogitsLoss()

    t0 = time.perf_counter()

    for _ in range(epochs):

        for xb, yb in loader:

            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()

            logits = model(xb)

            loss = criterion(
                logits,
                yb
            )

            if (
                old_params is not None
                and fisher is not None
            ):
                penalty = 0.0

                for name, param in model.named_parameters():

                    penalty += (
                        fisher[name]
                        *
                        (
                            param
                            -
                            old_params[name]
                        )**2
                    ).sum()

                loss = loss + (
                    ewc_lambda / 2.0
                ) * penalty

            loss.backward()
            optimizer.step()

    return time.perf_counter() - t0

@torch.no_grad()
def mlp_probs(model, X):

    model.eval()

    loader = make_loader(
        X,
        np.zeros(len(X), dtype=np.float32),
        shuffle=False
    )

    out = []

    for xb, _ in loader:
        xb = xb.to(DEVICE)

        out.append(
            torch.sigmoid(
                model(xb)
            ).cpu().numpy()
        )

    return np.concatenate(out)

def estimate_fisher(model, X, y):

    model.eval()

    loader = make_loader(X, y)

    criterion = nn.BCEWithLogitsLoss()

    fisher = {
        name: torch.zeros_like(
            param,
            device=DEVICE
        )
        for name, param in model.named_parameters()
    }

    total = 0

    for xb, yb in loader:

        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        model.zero_grad()

        loss = criterion(
            model(xb),
            yb
        )

        loss.backward()

        n = len(xb)
        total += n

        for name, param in model.named_parameters():

            if param.grad is not None:

                fisher[name] += (
                    param.grad.detach()**2
                ) * n

    for name in fisher:
        fisher[name] /= max(
            total,
            1
        )

    return fisher


In [19]:
# 19. Run EWC

ewc_model = SmallMLP(
    E1_X_train.shape[1]
).to(DEVICE)

EWC_LAMBDA = 100.0
EWC_EPOCHS = 5

old_params = None
fisher = None

ewc_rows = []
ewc_threshold = None

for i, (exp_name, data) in enumerate(
    stream_data.items(),
    start=1
):

    train_time = train_ewc(
        ewc_model,
        data["X_adapt"],
        data["y_adapt"],
        old_params=old_params,
        fisher=fisher,
        ewc_lambda=EWC_LAMBDA,
        epochs=EWC_EPOCHS,
        lr=1e-3
    )

    if i == 1:

        cal_probs = mlp_probs(
            ewc_model,
            data["X_cal"]
        )

        ewc_threshold, ewc_threshold_table, ewc_reason = select_threshold(
            data["y_cal"],
            cal_probs
        )

        ewc_threshold_table.to_csv(
            RESULTS / "ewc_threshold_selection.csv",
            index=False
        )

    for eval_name, eval_data in list(
        stream_data.items()
    )[:i]:

        probs = mlp_probs(
            ewc_model,
            eval_data["X_operational_eval"]
        )

        m = binary_metrics(
            eval_data["y_operational_eval"],
            probs,
            ewc_threshold
        )

        m.update({
            "method": "EWC_MLP",
            "after_experience": exp_name,
            "evaluated_experience": eval_name,
            "threshold": ewc_threshold,
            "train_time_sec": train_time,
            "rss_mb": rss_mb()
        })

        ewc_rows.append(m)

    fisher = estimate_fisher(
        ewc_model,
        data["X_adapt"],
        data["y_adapt"]
    )

    old_params = {
        name: param.detach().clone()
        for name, param in ewc_model.named_parameters()
    }

ewc_stream = pd.DataFrame(ewc_rows)

display(
    ewc_stream[
        ["method","after_experience",
         "evaluated_experience",
         "f1","recall","precision","fpr"]
    ].round(4)
)


,method,after_experience,evaluated_experience,f1,recall,precision,fpr
0,EWC_MLP,E1_initial_attack,E1_initial_attack,0.7182,0.693,0.7454,0.0365
1,EWC_MLP,E2_high_attack,E1_initial_attack,0.2359,1.000,0.1337,1.0000
2,EWC_MLP,E2_high_attack,E2_high_attack,0.7390,1.000,0.5861,1.0000
3,EWC_MLP,E3_attack_transition,E1_initial_attack,0.2359,1.000,0.1337,1.0000
4,EWC_MLP,E3_attack_transition,E2_high_attack,0.7390,1.000,0.5861,1.0000
5,EWC_MLP,E3_attack_transition,E3_attack_transition,0.6667,1.000,0.5000,1.0000
6,EWC_MLP,E4_generic_dominant,E1_initial_attack,0.2359,1.000,0.1337,1.0000
7,EWC_MLP,E4_generic_dominant,E2_high_attack,0.7390,1.000,0.5861,1.0000
8,EWC_MLP,E4_generic_dominant,E3_attack_transition,0.6667,1.000,0.5000,1.0000
9,EWC_MLP,E4_generic_dominant,E4_generic_dominant,0.6667,1.000,0.5000,1.0000


# 20. Full continual-learning matrices

For each method:

\[
M_{i,j}
\]

means:

> performance on experience `j` after learning through experience `i`.

Future cells must remain empty.


In [20]:
# 20. Build F1 and FPR matrices

def build_matrix(df, metric):

    rows = []

    for state in stream_data.keys():

        g = df[
            df["after_experience"] == state
        ]

        if g.empty:
            continue

        row = {
            "model_state": state
        }

        for _, r in g.iterrows():
            row[r["evaluated_experience"]] = r[metric]

        rows.append(row)

    return pd.DataFrame(rows).set_index(
        "model_state"
    )

naive_f1 = build_matrix(
    naive_stream,
    "f1"
)

naive_fpr = build_matrix(
    naive_stream,
    "fpr"
)

replay_f1 = build_matrix(
    replay_stream,
    "f1"
)

replay_fpr = build_matrix(
    replay_stream,
    "fpr"
)

ewc_f1 = build_matrix(
    ewc_stream,
    "f1"
)

ewc_fpr = build_matrix(
    ewc_stream,
    "fpr"
)

print("Naive F1:")
display(naive_f1.round(4))

print("Replay F1:")
display(replay_f1.round(4))

print("EWC F1:")
display(ewc_f1.round(4))


Naive F1:


,E1_initial_attack,E2_high_attack,E3_attack_transition,E4_generic_dominant,E5_stable_late
model_state,,,,,
E1_initial_attack,0.8577,NaN,NaN,NaN,NaN
E2_high_attack,0.0000,0.0,NaN,NaN,NaN
E3_attack_transition,0.0000,0.0,0.0,NaN,NaN
E4_generic_dominant,0.0000,0.0,0.0,0.0,NaN
E5_stable_late,0.0000,0.0,0.0,0.0,0.0


Replay F1:


,E1_initial_attack,E2_high_attack,E3_attack_transition,E4_generic_dominant,E5_stable_late
model_state,,,,,
E1_initial_attack,0.8577,NaN,NaN,NaN,NaN
E2_high_attack,0.6124,0.9922,NaN,NaN,NaN
E3_attack_transition,0.6434,0.9895,0.9877,NaN,NaN
E4_generic_dominant,0.6502,0.9880,0.9881,0.9879,NaN
E5_stable_late,0.6380,0.9893,0.9887,0.9879,0.9898


EWC F1:


,E1_initial_attack,E2_high_attack,E3_attack_transition,E4_generic_dominant,E5_stable_late
model_state,,,,,
E1_initial_attack,0.7182,NaN,NaN,NaN,NaN
E2_high_attack,0.2359,0.739,NaN,NaN,NaN
E3_attack_transition,0.2359,0.739,0.6667,NaN,NaN
E4_generic_dominant,0.2359,0.739,0.6667,0.6667,NaN
E5_stable_late,0.2359,0.739,0.6667,0.6667,0.6779


In [21]:
# 21. Save matrices

for name, matrix in [
    ("naive_f1_matrix.csv", naive_f1),
    ("naive_fpr_matrix.csv", naive_fpr),
    ("replay_f1_matrix.csv", replay_f1),
    ("replay_fpr_matrix.csv", replay_fpr),
    ("ewc_f1_matrix.csv", ewc_f1),
    ("ewc_fpr_matrix.csv", ewc_fpr)
]:
    matrix.to_csv(
        RESULTS / name
    )


# 22. Forgetting analysis

For each learned experience:

\[
F_i =
\max_t M_{t,i}
-
M_{T,i}
\]

Lower is better.

We calculate forgetting from F1.


In [22]:
# 22. Forgetting

def calculate_forgetting(matrix):

    rows = []

    for exp in stream_data.keys():

        vals = matrix[exp].dropna()

        if len(vals) == 0:
            continue

        best = vals.max()
        final = matrix.iloc[-1][exp]

        rows.append({
            "experience": exp,
            "best_f1": best,
            "final_f1": final,
            "forgetting": best - final
        })

    return pd.DataFrame(rows)

forget_naive = calculate_forgetting(
    naive_f1
)
forget_naive["method"] = "Blind_CL_LightGBM"

forget_replay = calculate_forgetting(
    replay_f1
)
forget_replay["method"] = "Replay_LightGBM"

forget_ewc = calculate_forgetting(
    ewc_f1
)
forget_ewc["method"] = "EWC_MLP"

forgetting = pd.concat(
    [
        forget_naive,
        forget_replay,
        forget_ewc
    ],
    ignore_index=True
)

display(forgetting.round(6))

forgetting.to_csv(
    RESULTS / "forgetting_analysis.csv",
    index=False
)


,experience,best_f1,final_f1,forgetting,method
0,E1_initial_attack,0.857659,0.000000,0.857659,Blind_CL_LightGBM
1,E2_high_attack,0.000000,0.000000,0.000000,Blind_CL_LightGBM
2,E3_attack_transition,0.000000,0.000000,0.000000,Blind_CL_LightGBM
3,E4_generic_dominant,0.000000,0.000000,0.000000,Blind_CL_LightGBM
4,E5_stable_late,0.000000,0.000000,0.000000,Blind_CL_LightGBM
5,E1_initial_attack,0.857659,0.638045,0.219614,Replay_LightGBM
6,E2_high_attack,0.992219,0.989343,0.002877,Replay_LightGBM
7,E3_attack_transition,0.988662,0.988662,0.000000,Replay_LightGBM
8,E4_generic_dominant,0.987868,0.987861,0.000007,Replay_LightGBM
9,E5_stable_late,0.989783,0.989783,0.000000,Replay_LightGBM


# 23. Final stream retention and FPR

After E5, evaluate the final model state on every experience.

This produces:

- final F1 retention;
- final recall;
- final FPR.

This is more informative than final E5 F1 alone.


In [23]:
# 23. Final retention

retention_rows = []

for method, df in [
    ("Blind_CL_LightGBM", naive_stream),
    ("Replay_LightGBM", replay_stream),
    ("EWC_MLP", ewc_stream)
]:

    final_state = list(stream_data.keys())[-1]

    g = df[
        df["after_experience"] == final_state
    ]

    for _, r in g.iterrows():

        retention_rows.append({
            "method": method,
            "experience": r["evaluated_experience"],
            "final_f1": r["f1"],
            "final_recall": r["recall"],
            "final_precision": r["precision"],
            "final_fpr": r["fpr"],
            "final_balanced_accuracy": r["balanced_accuracy"]
        })

# Static
for _, r in static_stream.iterrows():

    retention_rows.append({
        "method": "Static_LightGBM",
        "experience": r["evaluated_experience"],
        "final_f1": r["f1"],
        "final_recall": r["recall"],
        "final_precision": r["precision"],
        "final_fpr": r["fpr"],
        "final_balanced_accuracy": r["balanced_accuracy"]
    })

final_retention = pd.DataFrame(
    retention_rows
)

display(
    final_retention.round(6)
)

final_retention.to_csv(
    RESULTS / "final_stream_retention.csv",
    index=False
)


,method,experience,final_f1,final_recall,final_precision,final_fpr,final_balanced_accuracy
0,Blind_CL_LightGBM,E1_initial_attack,0.000000,0.000000,0.000000,0.000000,0.500000
1,Blind_CL_LightGBM,E2_high_attack,0.000000,0.000000,0.000000,0.000000,0.500000
2,Blind_CL_LightGBM,E3_attack_transition,0.000000,0.000000,0.000000,0.000000,0.500000
3,Blind_CL_LightGBM,E4_generic_dominant,0.000000,0.000000,0.000000,0.000000,0.500000
4,Blind_CL_LightGBM,E5_stable_late,0.000000,0.000000,0.000000,0.000000,0.500000
5,Replay_LightGBM,E1_initial_attack,0.638045,0.992893,0.470054,0.172792,0.910050
6,Replay_LightGBM,E2_high_attack,0.989343,0.993220,0.985495,0.020700,0.986260
7,Replay_LightGBM,E3_attack_transition,0.988662,0.998204,0.979300,0.021100,0.988552
8,Replay_LightGBM,E4_generic_dominant,0.987861,0.997719,0.978196,0.022239,0.987740
9,Replay_LightGBM,E5_stable_late,0.989783,0.999050,0.980687,0.020700,0.989175


# 24. Resource comparison


In [24]:
# 24. Resource summaries

resource_rows = [
    {
        "method": "Static_LightGBM",
        "total_training_time_sec": static_train_time,
        "max_rss_mb": np.nan,
        "max_replay_size": 0
    },
    {
        "method": "Blind_CL_LightGBM",
        "total_training_time_sec": naive_stream["train_time_sec"].sum(),
        "max_rss_mb": naive_stream["rss_mb"].max(),
        "max_replay_size": 0
    },
    {
        "method": "Replay_LightGBM",
        "total_training_time_sec": replay_stream["train_time_sec"].sum(),
        "max_rss_mb": replay_stream["rss_mb"].max(),
        "max_replay_size": replay_stream["replay_size"].max()
    },
    {
        "method": "EWC_MLP",
        "total_training_time_sec": ewc_stream["train_time_sec"].sum(),
        "max_rss_mb": ewc_stream["rss_mb"].max(),
        "max_replay_size": 0
    }
]

resource_summary = pd.DataFrame(
    resource_rows
)

display(resource_summary.round(4))

resource_summary.to_csv(
    RESULTS / "resource_summary.csv",
    index=False
)


,method,total_training_time_sec,max_rss_mb,max_replay_size
0,Static_LightGBM,10.8573,NaN,0
1,Blind_CL_LightGBM,8.8503,1202.0234,0
2,Replay_LightGBM,34.1271,1240.3750,8000
3,EWC_MLP,29.1047,1352.6055,0


# 25. Final independent temporal test

Thresholds are frozen from E1 calibration.

The final test is not used for tuning.


In [25]:
# 25. Final temporal test

test_rows = []

# Static
probs = static_model.predict_proba(
    preprocessor.transform(
        X_test_raw
    ).astype(np.float32)
)[:,1]

m = binary_metrics(
    y_test,
    probs,
    static_threshold
)

m.update({
    "method": "Static_LightGBM",
    "threshold": static_threshold
})

test_rows.append(m)

# Naive
probs = naive_model.predict_proba(
    preprocessor.transform(
        X_test_raw
    ).astype(np.float32)
)[:,1]

m = binary_metrics(
    y_test,
    probs,
    naive_threshold
)

m.update({
    "method": "Blind_CL_LightGBM",
    "threshold": naive_threshold
})

test_rows.append(m)

# Replay
probs = replay_model.predict_proba(
    preprocessor.transform(
        X_test_raw
    ).astype(np.float32)
)[:,1]

m = binary_metrics(
    y_test,
    probs,
    replay_threshold
)

m.update({
    "method": "Replay_LightGBM",
    "threshold": replay_threshold
})

test_rows.append(m)

# EWC
X_test_proc = preprocessor.transform(
    X_test_raw
).astype(np.float32)

probs = mlp_probs(
    ewc_model,
    X_test_proc
)

m = binary_metrics(
    y_test,
    probs,
    ewc_threshold
)

m.update({
    "method": "EWC_MLP",
    "threshold": ewc_threshold
})

test_rows.append(m)

final_test = pd.DataFrame(
    test_rows
)

display(
    final_test[
        [
            "method",
            "threshold",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "fpr",
            "specificity",
            "balanced_accuracy",
            "roc_auc",
            "pr_auc"
        ]
    ].round(6)
)

final_test.to_csv(
    RESULTS / "final_temporal_test_comparison.csv",
    index=False
)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,method,threshold,accuracy,precision,recall,f1,fpr,specificity,balanced_accuracy,roc_auc,pr_auc
0,Static_LightGBM,0.45,0.815236,0.906697,0.740647,0.815303,0.093378,0.906622,0.823634,0.935382,0.942424
1,Blind_CL_LightGBM,0.45,0.449400,0.000000,0.000000,0.000000,0.000000,1.000000,0.500000,0.500000,0.550600
2,Replay_LightGBM,0.45,0.817762,0.752372,0.997243,0.857671,0.402135,0.597865,0.797554,0.976015,0.981790
3,EWC_MLP,0.38,0.550600,0.550600,1.000000,0.710177,1.000000,0.000000,0.500000,0.556449,0.580029


# 26. Aggregate results

The benchmark is evaluated across several dimensions.

A method is not declared superior merely because it has the highest F1.

We inspect:

- mean final stream F1;
- mean final stream FPR;
- mean forgetting;
- final independent-test F1;
- final independent-test FPR;
- total training cost;
- replay memory.


In [26]:
# 26. Aggregate comparison

aggregate_rows = []

for method in [
    "Static_LightGBM",
    "Blind_CL_LightGBM",
    "Replay_LightGBM",
    "EWC_MLP"
]:

    g = final_retention[
        final_retention["method"] == method
    ]

    row = {
        "method": method,
        "mean_final_stream_f1": g["final_f1"].mean(),
        "mean_final_stream_fpr": g["final_fpr"].mean(),
        "mean_final_stream_balanced_accuracy":
            g["final_balanced_accuracy"].mean()
    }

    fg = forgetting[
        forgetting["method"] == method
    ]

    row["mean_forgetting"] = (
        fg["forgetting"].mean()
        if len(fg)
        else 0.0
    )

    tr = final_test[
        final_test["method"] == method
    ].iloc[0]

    row["final_test_f1"] = tr["f1"]
    row["final_test_recall"] = tr["recall"]
    row["final_test_fpr"] = tr["fpr"]
    row["final_test_balanced_accuracy"] = tr["balanced_accuracy"]

    rr = resource_summary[
        resource_summary["method"] == method
    ].iloc[0]

    row["total_training_time_sec"] = rr[
        "total_training_time_sec"
    ]

    row["max_replay_size"] = rr[
        "max_replay_size"
    ]

    aggregate_rows.append(row)

aggregate = pd.DataFrame(
    aggregate_rows
)

display(
    aggregate.round(6)
)

aggregate.to_csv(
    RESULTS / "continual_learning_aggregate_results.csv",
    index=False
)


,method,mean_final_stream_f1,mean_final_stream_fpr,mean_final_stream_balanced_accuracy,mean_forgetting,final_test_f1,final_test_recall,final_test_fpr,final_test_balanced_accuracy,total_training_time_sec,max_replay_size
0,Static_LightGBM,0.867282,0.010514,0.889454,0.000000,0.815303,0.740647,0.093378,0.823634,10.857324,0
1,Blind_CL_LightGBM,0.000000,0.000000,0.500000,0.171532,0.000000,0.000000,0.000000,0.500000,8.850333,0
2,Replay_LightGBM,0.918739,0.051506,0.972355,0.044499,0.857671,0.997243,0.402135,0.797554,34.127056,8000
3,EWC_MLP,0.597225,1.000000,0.500000,0.096467,0.710177,1.000000,1.000000,0.500000,29.104665,0


# 27. Protocol integrity checks

These must pass before Phase 2B is accepted.

Checks include:

- E1 train/calibration/evaluation all contain both classes;
- benign reference contains only benign traffic;
- benign reference is never included in adaptation arrays;
- final test is independent;
- thresholds are valid;
- future experience cells are absent from continual matrices.


In [27]:
# 27. Integrity checks

# E1 class validity
assert set(np.unique(E1_y_train)) == {0, 1}
assert set(np.unique(E1_y_cal)) == {0, 1}
assert set(np.unique(E1_y_attack_eval)) == {0, 1}

# Benign reference
assert np.all(y_benign_ref == 0)

# Attack adaptation for E2-E5
for name in list(stream_data.keys())[1:]:
    assert np.all(
        stream_data[name]["y_adapt"] == 1
    )
    assert np.all(
        stream_data[name]["y_attack_eval"] == 1
    )

# Evaluation must contain both classes
for name, data in stream_data.items():
    assert set(
        np.unique(data["y_operational_eval"])
    ) == {0, 1}

# Thresholds
for t in [
    static_threshold,
    naive_threshold,
    replay_threshold,
    ewc_threshold
]:
    assert 0.0 < t < 1.0

# Future cells must be NaN
for matrix in [
    naive_f1,
    replay_f1,
    ewc_f1,
    naive_fpr,
    replay_fpr,
    ewc_fpr
]:
    states = list(matrix.index)

    for i, state in enumerate(states):
        future = states[i+1:]

        for exp in future:
            assert pd.isna(
                matrix.loc[state, exp]
            )

print("ALL PHASE-2B-v3 INTEGRITY CHECKS PASSED.")


ALL PHASE-2B-v3 INTEGRITY CHECKS PASSED.


# 28. Save artifacts


In [28]:
# 28. Save models and protocol

joblib.dump(
    static_model,
    ARTIFACTS / "static_lightgbm.joblib"
)

joblib.dump(
    naive_model,
    ARTIFACTS / "blind_continual_lightgbm.joblib"
)

joblib.dump(
    replay_model,
    ARTIFACTS / "replay_lightgbm.joblib"
)

torch.save(
    ewc_model.state_dict(),
    ARTIFACTS / "ewc_mlp_state.pt"
)

# Save benign reference metadata, not raw data.
benign_reference_metadata = {
    "source_segments": [1, 2],
    "source_size": int(len(benign_reference_source)),
    "evaluation_reference_size": int(BENIGN_REFERENCE_SIZE),
    "random_state": SEED,
    "used_for_training": False,
    "used_for_replay": False,
    "used_for_threshold_calibration": False
}

with open(
    RESULTS / "benign_reference_protocol.json",
    "w"
) as f:
    json.dump(
        benign_reference_metadata,
        f,
        indent=2
    )

protocol = {
    "dataset": "lacg030175/UNSW-NB15",
    "config": "temporal",
    "experiences": EXPERIENCES,
    "e1_split": {
        "training": 0.56,
        "calibration": 0.14,
        "attack_evaluation": 0.30,
        "method": "stratified"
    },
    "later_experience_split": {
        "adaptation": 0.70,
        "attack_evaluation": 0.30,
        "method": "chronological"
    },
    "operational_evaluation": (
        "current attack evaluation + fixed benign reference "
        "from Segments 1-2"
    ),
    "threshold": {
        "source": "E1 calibration only",
        "fpr_constraint": 0.10,
        "range": [0.05, 0.95],
        "frozen_after_e1": True
    },
    "methods": [
        "Static_LightGBM",
        "Blind_CL_LightGBM",
        "Replay_LightGBM",
        "EWC_MLP"
    ],
    "replay_per_experience": REPLAY_PER_EXPERIENCE,
    "ewc_lambda": EWC_LAMBDA,
    "ewc_epochs": EWC_EPOCHS,
    "final_test_used_for_training": False,
    "final_test_used_for_threshold_tuning": False,
    "seed": SEED,
    "interpretation": (
        "Operational evaluation is a controlled stress test. "
        "Its fixed benign reference is not the empirical class ratio "
        "of the original attack-heavy segments."
    )
}

with open(
    RESULTS / "phase2b_v3_protocol.json",
    "w"
) as f:
    json.dump(
        protocol,
        f,
        indent=2
    )

bundle = shutil.make_archive(
    str(BASE / "phase2b_v3_artifacts"),
    "zip",
    root_dir=BASE
)

print("Created:", bundle)

print("\nKey outputs:")
for p in sorted(RESULTS.glob("*")):
    print(" -", p)


Created: /content/carc_ids_phase2b_v3/phase2b_v3_artifacts.zip

Key outputs:
 - /content/carc_ids_phase2b_v3/results/benign_reference_protocol.json
 - /content/carc_ids_phase2b_v3/results/continual_learning_aggregate_results.csv
 - /content/carc_ids_phase2b_v3/results/ewc_f1_matrix.csv
 - /content/carc_ids_phase2b_v3/results/ewc_fpr_matrix.csv
 - /content/carc_ids_phase2b_v3/results/ewc_threshold_selection.csv
 - /content/carc_ids_phase2b_v3/results/experience_summary.csv
 - /content/carc_ids_phase2b_v3/results/final_stream_retention.csv
 - /content/carc_ids_phase2b_v3/results/final_temporal_test_comparison.csv
 - /content/carc_ids_phase2b_v3/results/forgetting_analysis.csv
 - /content/carc_ids_phase2b_v3/results/naive_f1_matrix.csv
 - /content/carc_ids_phase2b_v3/results/naive_fpr_matrix.csv
 - /content/carc_ids_phase2b_v3/results/naive_threshold_selection.csv
 - /content/carc_ids_phase2b_v3/results/phase2b_v3_protocol.json
 - /content/carc_ids_phase2b_v3/results/replay_f1_matrix.csv


# 29. Decision gate

Do not move to Phase 2C until the outputs are inspected.

## Phase 2B is accepted if

- static detection is sensible;
- blind adaptation demonstrates interpretable failure or trade-off;
- replay is measurable as a conventional protection strategy;
- EWC is interpretable as a neural reference;
- FPR is measurable for every stream experience;
- forgetting is measurable;
- resource costs are measurable;
- final temporal-test results are plausible;
- all integrity checks pass.

## The key comparison for Phase 2C

We will compare the security/resource trade-off:

```text
Static
   vs
Blind adaptation
   vs
Replay
   vs
Resource-aware selective adaptation
```

The proposed controller must not be justified merely because it is more complex.

It must demonstrate that **selective adaptation can avoid unnecessary updates while retaining detection performance and controlling false positives/resources**.

Only after that should the IPS response layer and local LLM be introduced.
